# Thesis pipeline: every step in one notebook
**Cross-network intrusion detection on NetFlow v3** · repository: `mirmumin/netflow-crossdataset-ids`

Running this notebook from top to bottom reproduces every result of the thesis. Each step starts with a short note on **what it does, why, how long it takes and where its output goes**.

| Step | Task | Runtime | Approx. time |
|---|---|---|---|
| 0 | Setup: Drive, code, packages | any | 2 min |
| 1 | Data preparation (stratified subsample) | CPU | 20-30 min first time, then 1 min |
| 2 | Exploratory analysis (EDA), optional | CPU | 2 min |
| 3 | Baselines: RF, XGBoost, SVM, LR, KNN, Ensemble | CPU | 1-1.5 h |
| 4 | Few-shot fusion (few labelled target flows) | **GPU** | 20-30 min |
| 5 | Benign-only target (proposed Fusion-B), 5 seeds | **GPU** | 35-40 min per seed |
| 6 | Statistics (Wilcoxon) and paper figures | any | 1 min |
| 7 | Word report and key tables | any | 1 min |

**All results are written straight to Google Drive** (`MyDrive/thesis_data/results/`), so nothing is lost if Colab disconnects. Re-running a cell continues where it stopped.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Steps 4-5 need the GPU; the other steps also run on it.

## Step 0 · Setup
**What it does:** mounts Google Drive, fetches the code from GitHub (the repository is private, so a token stored in Colab Secrets as `GH_TOKEN` is used) and installs the extra packages.

**One-time preparation:** click the key icon on the left, add a secret named `GH_TOKEN` holding a read-only GitHub token, and switch on "Notebook access".

When Drive asks for permissions, choose "Select all", otherwise mounting fails.

In [ ]:
# --- Step 0: setup -------------------------------------------------------------
import os, subprocess
from google.colab import drive, userdata

# 1) Connect Google Drive (data and results live there permanently)
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/thesis_data'      # folder with the 4 zips / processed data
RES   = f'{DRIVE}/results'                        # all result files are written here
os.makedirs(RES, exist_ok=True)

# 2) Get the code. The repo is private, so we use the read-only token saved in Colab Secrets.
REPO = '/content/netflow-crossdataset-ids'
BRANCH = 'main'           # code version (git branch) to run
token = userdata.get('GH_TOKEN')
if not os.path.exists(REPO):
    r = subprocess.run(['git', 'clone', f'https://{token}@github.com/mirmumin/netflow-crossdataset-ids.git', REPO],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***'))   # never print the token
%cd /content/netflow-crossdataset-ids
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!git log --oneline -1                                   # shows which code version is running

# 3) Packages. Colab already has pandas, scikit-learn, xgboost; we add the rest.
!pip install -q tabpfn==9.2.0 python-docx pyyaml pyarrow

# 4) Show whether a GPU is available (needed for steps 4 and 5)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU only)')

## Step 1 · Data preparation
**What it does:** takes about 500,000 flows from each of the four NetFlow v3 datasets (about 13 GB in total) by **stratified subsampling**: every attack class keeps its share, small classes are kept whole, and at least 100,000 benign flows are kept.

**Why:** the full data does not fit in free Colab memory; 500k flows per dataset is large and representative.

**Output:** `thesis_data/processed/*.parquet`. Once created, later sessions only copy these files (about 1 minute).

Code: `scripts/prepare_data.py`, `src/data.py`

In [ ]:
# --- Step 1: data preparation ---------------------------------------------------
import glob, zipfile, shutil
PROC_DRIVE = f'{DRIVE}/processed'
os.makedirs('data/processed', exist_ok=True)

if len(glob.glob(f'{PROC_DRIVE}/*.parquet')) == 4:
    # Already done in an earlier session: just copy the 4 small parquet files
    !cp {PROC_DRIVE}/*.parquet {PROC_DRIVE}/*.csv {PROC_DRIVE}/*.json data/processed/ 2>/dev/null
    print('Processed data copied from Drive.')
else:
    # First time: unzip the 4 UQ zip files from Drive into data/raw/
    os.makedirs('data/raw', exist_ok=True)
    for z in glob.glob(f'{DRIVE}/*.zip'):
        name = os.path.splitext(os.path.basename(z))[0]
        print('unzipping', name)
        zipfile.ZipFile(z).extractall(f'data/raw/{name}')
    # Stratified subsample (~500k flows per dataset) -> data/processed/*.parquet
    !python scripts/prepare_data.py
    os.makedirs(PROC_DRIVE, exist_ok=True)
    !cp data/processed/* {PROC_DRIVE}/

# Class balance of each subsample (rows = attack classes)
import pandas as pd
pd.read_csv('data/processed/class_summary.csv')

## Step 2 · Exploratory analysis (EDA), optional
**What it does:** measures how much each feature's distribution differs between networks (Kolmogorov-Smirnov statistic: 0 = identical, 1 = completely different) and draws a heatmap.

**Why:** this figure shows why a model trained on one network fails on another; it is the motivation of the thesis.

**Output:** `results/figures/ks_shift_heatmap.png`, `attack_rate.png`

Code: `notebooks/01_eda.ipynb`

In [ ]:
# --- Step 2 (optional): run the EDA notebook headless and keep its figures --------
!jupyter nbconvert --to notebook --execute notebooks/01_eda.ipynb --output 01_eda_run.ipynb
os.makedirs(f'{RES}/figures', exist_ok=True)
!cp results/figures/*.png {RES}/figures/ && cp results/ks_shift_benign.csv {RES}/
from IPython.display import Image, display
display(Image(f'{RES}/figures/ks_shift_heatmap.png', width=600))

## Step 3 · Baselines: in-dataset and cross-dataset
**What it does:** trains five classical models (RF, XGBoost, SVM, LR, KNN) and their soft-voting **Ensemble** (average attack probability of the five). Each model is trained on one network and tested on the test split of all four networks.
- Tested on the same network: **in-dataset** result (the supervisor's table).
- Tested on another network: **cross-dataset** result (4×4 = 12 pairs). This is where the generalisation gap shows.

**Result (seed 42):** in-dataset F1 84-100%, but mean cross-dataset F1 only 18-33%.

**Time:** 1-1.5 h. Finished runs are skipped, so an interrupted run can simply be restarted. Add `--all-seeds` for the final 5-seed numbers (5× the time).

Code: `scripts/run_baselines.py`, `scripts/summarise_baselines.py`, `src/models.py`

In [ ]:
# --- Step 3: classical baselines (in-dataset + 4x4 cross-dataset) ----------------
# Results are appended to {RES}/baselines_raw.csv as each model finishes.
!python scripts/run_baselines.py --feature-sets v3 --results-dir {RES}

# Turn the raw rows into paper tables (supervisor's table = table_models.md)
!python scripts/summarise_baselines.py {RES}

## Step 4 · Few-shot fusion (k = 100 / 1000 labelled target flows)
**Setting:** the new (target) network has only k labelled flows, benign and attack; the other three (source) networks are fully labelled.

**Compared methods:**
- `XGB-src`: XGBoost trained on the sources only, no target labels (zero-shot)
- `XGB-k`, `TabPFN-k`: trained on the k target labels only
- `Fusion-k`: TabPFN on the k target labels, with the source XGBoost score as an extra feature

**Result (seed 42):** Fusion ≈ TabPFN, no gain. With just 100 target labels F1 is already 84-99%, so source knowledge does not help in this setting (in line with Rayamajhi, 2026). This motivates Step 5.

Code: `scripts/run_fusion.py`, `src/fusion.py`

In [ ]:
# --- Step 4: few-shot fusion (needs GPU for TabPFN) ------------------------------
!python scripts/run_fusion.py --results-dir {RES}

## Step 5 · Benign-only target (proposed Fusion-B)
**Setting (realistic):** the operator of a new network can easily record its **normal traffic**, but has **no labelled attacks**. Knowledge of attacks can only come from other networks.

**Compared methods:**
- `XGB-src`: XGBoost trained on the source networks only
- `IForest-k`: Isolation Forest fitted on the k target benign flows (anomaly detection, no source)
- `XGB-pool-k`: XGBoost on source data plus the k target benign flows
- `Fusion-B-k` (**proposed**): TabPFN whose context holds the k target benign flows and source attack flows, plus two extra signals: the source XGBoost score and an anomaly score
- `XGB-src-BQN-k`: XGBoost on features aligned to each network's own benign traffic (quantile mapping)
- `Fusion-B-v2-k` (variant of the proposal): Fusion-B on those benign-aligned features

**Fairness rules:**
- The source attack flows placed in the context are held out from the source XGBoost's training data.
- Every method uses the same decision threshold: the 95th percentile of its cross-fitted scores on the known benign flows (`rule = benign-q95` in the results).

**Time:** about 35-40 min per seed on a T4 GPU. Use `--all-seeds` (5 seeds) for the paper. Finished parts are skipped.

Code: `scripts/run_benign_only.py`, `src/align.py`

In [ ]:
# --- Step 5: benign-only target setting, proposed Fusion-B v2 (needs GPU) --------
# --all-seeds = seeds 42-46 for the paper (drop it for a quick single-seed run)
!python scripts/run_benign_only.py --results-dir {RES} --all-seeds

## Step 6 · Statistics (Wilcoxon) and paper figures
**What it does:** from the 5-seed results of Step 5:
- **Wilcoxon signed-rank test:** are the proposed methods (Fusion-B and Fusion-B-v2) really better than each other method, or is it chance? Each pair is one (network, k, seed) run, 40 pairs in total; p-values are Holm-corrected because several comparisons are made.
- **Figures:** F1 and AUROC bar charts per network (mean ± SD).

**Result:** the Fusion-B family is clearly better than target-only IForest and zero-shot XGBoost (p < 0.001); the difference between v2 and Fusion-B is not significant.

**Output:** `results/stats_benign_only.csv`, `results/figures/benign_only_f1.png/.pdf`, `benign_only_auroc.png/.pdf`

Code: `scripts/stats_benign_only.py`, `scripts/plot_benign_only.py`

In [ ]:
# --- Step 6: statistics + paper figures for the benign-only setting -----------
# Paired Wilcoxon tests: Fusion-B and Fusion-B-v2 vs every other method (F1 and AUROC)
!python scripts/stats_benign_only.py {RES}

# Bar charts (mean +- SD over seeds), saved as PNG and PDF for the paper
!python scripts/plot_benign_only.py {RES}
!python scripts/plot_paper_figures.py {RES}   # heatmap, Setting B, k effect, threshold, Wilcoxon, runtime
from IPython.display import Image, display
display(Image(f'{RES}/figures/benign_only_f1.png', width=900))
display(Image(f'{RES}/figures/benign_only_auroc.png', width=900))
import glob
for f in sorted(glob.glob(f'{RES}/figures/fig_*.png')):
    print(os.path.basename(f))
    display(Image(f, width=900))


## Step 7 · Word report and tables
**What it does:** collects all results so far into `report.docx` (setup, supervisor's table, cross-dataset, fusion, benign-only) and shows the key tables below.

**Output:** `MyDrive/thesis_data/results/report.docx`, which opens in Word or Google Docs.

Code: `scripts/make_report.py`

In [ ]:
# --- Step 7: Word report + show the key tables here -------------------------------
!python scripts/make_report.py {RES}

from IPython.display import Markdown, display
if os.path.exists(f'{RES}/table_models.md'):                 # supervisor's table (in-dataset)
    display(Markdown(open(f'{RES}/table_models.md').read()))
for name in ['table_cross_summary.csv', 'fusion_summary.csv', 'benign_only_summary.csv', 'stats_benign_only.csv']:
    path = f'{RES}/{name}'
    if os.path.exists(path):
        display(Markdown(f'**{name}**'))
        display(pd.read_csv(path))